## External table using absolute path to HDFS file system. '/user/user025320' is the HDFS file system

In [1]:
from pyspark.sql import SparkSession
import getpass

username = getpass.getuser()

In [2]:
spark = SparkSession.builder. \
config('spark.ui.port','0'). \
config("spark.sql.warehouse.dir", f"/user/{username}/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()

In [3]:
read_df = spark.read.format("csv"). \
option("inferSchema", "true"). \
load("external_file/retail_db_orders_part-00000")

In [4]:
read_df.show(10)

+---+--------------------+-----+---------------+
|_c0|                 _c1|  _c2|            _c3|
+---+--------------------+-----+---------------+
|  1|2013-07-25 00:00:...|11599|         CLOSED|
|  2|2013-07-25 00:00:...|  256|PENDING_PAYMENT|
|  3|2013-07-25 00:00:...|12111|       COMPLETE|
|  4|2013-07-25 00:00:...| 8827|         CLOSED|
|  5|2013-07-25 00:00:...|11318|       COMPLETE|
|  6|2013-07-25 00:00:...| 7130|       COMPLETE|
|  7|2013-07-25 00:00:...| 4530|       COMPLETE|
|  8|2013-07-25 00:00:...| 2911|     PROCESSING|
|  9|2013-07-25 00:00:...| 5657|PENDING_PAYMENT|
| 10|2013-07-25 00:00:...| 5648|PENDING_PAYMENT|
+---+--------------------+-----+---------------+
only showing top 10 rows



In [5]:
from pyspark.sql.functions import col
df_renamed = read_df.select(
    col('_c0').alias('order_id'),
    col('_c1').alias('order_date'),
    col('_c2').alias('customer_id') ,
    col('_c3').alias('order_status')
)
df_renamed.show(5)

+--------+--------------------+-----------+---------------+
|order_id|          order_date|customer_id|   order_status|
+--------+--------------------+-----------+---------------+
|       1|2013-07-25 00:00:...|      11599|         CLOSED|
|       2|2013-07-25 00:00:...|        256|PENDING_PAYMENT|
|       3|2013-07-25 00:00:...|      12111|       COMPLETE|
|       4|2013-07-25 00:00:...|       8827|         CLOSED|
|       5|2013-07-25 00:00:...|      11318|       COMPLETE|
+--------+--------------------+-----------+---------------+
only showing top 5 rows



In [6]:
df_renamed.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- order_date: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_status: string (nullable = true)



In [7]:
df_renamed.schema

StructType(List(StructField(order_id,IntegerType,true),StructField(order_date,StringType,true),StructField(customer_id,IntegerType,true),StructField(order_status,StringType,true)))

In [8]:
df_renamed.dtypes

[('order_id', 'int'),
 ('order_date', 'string'),
 ('customer_id', 'int'),
 ('order_status', 'string')]

In [9]:
df_renamed.columns

['order_id', 'order_date', 'customer_id', 'order_status']

In [10]:
df_renamed.count()

68883

In [11]:
df_renamed.createOrReplaceTempView('order_view')

In [21]:
spark.sql("""
    CREATE TABLE big_data_025320.order_external
    USING PARQUET
    LOCATION '/user/itv025320/external_tables/order'
    AS SELECT 
        order_id,
        CAST(order_date AS TIMESTAMP) as order_date,
        customer_id,
        order_status
    FROM order_view
""")

""


In [13]:
#spark.sql("DROP TABLE IF EXISTS big_data_025320.order_external")

In [22]:
spark.sql("describe extended big_data_025320.order_external").show(truncate=False)

+----------------------------+------------------------------------------------------------------+-------+
|col_name                    |data_type                                                         |comment|
+----------------------------+------------------------------------------------------------------+-------+
|order_id                    |int                                                               |null   |
|order_date                  |timestamp                                                         |null   |
|customer_id                 |int                                                               |null   |
|order_status                |string                                                            |null   |
|                            |                                                                  |       |
|# Detailed Table Information|                                                                  |       |
|Database                    |big_data_025320 

#### **External** Table created in the HDFS file system
```
[username@g02 data]$ hdfs dfs -ls -R /user/username/external_tables/
drwxr-xr-x   - username supergroup          0 2026-03-30 05:41 /user/username/external_tables/order
-rw-r--r--   3 username supergroup          0 2026-03-30 05:41 /user/username/external_tables/order/_SUCCESS
-rw-r--r--   3 username supergroup     487478 2026-03-30 05:41 /user/username/external_tables/order/part-00000-527264b6-6f59-4738-8ca9-853669a79483-c000.snappy.parquet
```

In [23]:
#spark.sql("DROP TABLE IF EXISTS big_data_025320.order_external")

In [16]:
#spark.sql("describe formatted big_data_025320.order_external").show(truncate=False)

In [19]:
spark.sql("show databases").filter("namespace = 'big_data_025320'")

namespace
big_data_025320


In [24]:
# Now Let's see if the table is dropped then what happens

spark.sql("DROP TABLE IF EXISTS big_data_025320.order_external")

""


In [25]:
spark.sql("DESCRIBE TABLE big_data_025320.order_external")

AnalysisException: Table or view not found for 'DESCRIBE TABLE': big_data_025320.order_external; line 1 pos 0;
'DescribeRelation false, [col_name#241, data_type#242, comment#243]
+- 'UnresolvedTableOrView [big_data_025320, order_external], DESCRIBE TABLE, true


##### Let's check **External table in HDFS** storage now. 
```
[username@g02 data]$ hdfs dfs -ls -R /user/username/external_tables/
drwxr-xr-x   - username supergroup          0 2026-03-30 05:41 /user/username/external_tables/order
-rw-r--r--   3 username supergroup          0 2026-03-30 05:41 /user/username/external_tables/order/_SUCCESS
-rw-r--r--   3 username supergroup     487478 2026-03-30 05:41 /user/username/external_tables/order/part-00000-527264b6-6f59-4738-8ca9-853669a79483-c000.snappy.parquet
```

##### Files are not deleted as they're external.

### 1. /user/{username}/external_tables/order
#### where /user/{username} is the HDFS file system while /home/{username} is the local (driver/gateway node) file system. **spark running on cluster simply has no way of reaching a local file system.**

### 2. **external_tables/order** path isn't created beforehand. spark creates it. So it need not be created explicitely beforehand

### 3. If there are multiple external tables then:

```
/user/{username}/external_tables/
    ├── orders/
    │    ├── part-00001.parquet
    │    └── part-00002.parquet
    └── customers/
         ├── part-00001.parquet
         └── part-00002.parquet
```
#### By using /orders at the end of the path, you are ensuring Isolation. It keeps your data organized, prevents different tables from reading each other's files, and ensures that DROP or INSERT operations only affect the data belonging to that specific table.

### 4. **IF NOT EXISTS** in CTAS
```
spark.sql("""
    CREATE TABLE IF NOT EXISTS big_data_025320.order_external
    USING PARQUET
    LOCATION '/user/itv025320/external_tables/orders'  -- Points to HDFS
    AS SELECT 
        order_id,
        CAST(order_date AS TIMESTAMP) as order_date,
        customer_id,
        order_status
    FROM order_view
""")
```
#### IF NOT EXISTS is a "silent skipper." If a table with that name already exists in the Metastore, Spark simply ignores entire command—including the new LOCATION provided—and moves on without giving an error.

In [26]:
spark.stop()